# Clase 2 — Aritmética Modular y Teoría de Números

**Curso:** Criptografía Aplicada  
**Temas:** Módulo · Congruencias · Algoritmo de Euclides extendido · Inversos modulares · Teorema Chino del Resto · Pequeño Teorema de Fermat · Teorema de Euler · Exponenciación modular · Tests de primalidad · Aplicación a RSA  

---

> *"Las matemáticas son la reina de las ciencias, y la teoría de números es la reina de las matemáticas."*  
> — Carl Friedrich Gauss


## Tabla de Contenidos

1. [Módulo y Congruencias](#1-modulo)
2. [Propiedades de la Aritmética Modular](#2-propiedades)
3. [Máximo Común Divisor y Algoritmo de Euclides](#3-euclides)
4. [Algoritmo de Euclides Extendido](#4-euclides-ext)
5. [Inversos Modulares](#5-inversos)
6. [Exponenciación Modular Rápida](#6-expmod)
7. [Teorema Chino del Resto](#7-crt)
8. [Pequeño Teorema de Fermat](#8-fermat)
9. [Función Phi de Euler y Teorema de Euler](#9-euler)
10. [Números Primos y Tests de Primalidad](#10-primalidad)
11. [Aplicación: RSA desde cero](#11-rsa)
12. [Ejercicios y Desafíos](#12-ejercicios)
13. [Referencias](#13-referencias)


In [ ]:
# ── Importaciones para toda la clase ──────────────────────────────────────────
import math
import random
import sympy
from typing import List, Tuple

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
print('Librerías cargadas correctamente.')

---
## 1. Módulo y Congruencias <a id='1-modulo'></a>

### 1.1 La operación módulo

Dado un entero $a$ y un entero positivo $n$, el **residuo** de dividir $a$ entre $n$ se escribe:

$$a \bmod n = r \quad \text{donde} \quad a = qn + r, \quad 0 \le r < n$$

El valor $n$ recibe el nombre de **módulo**. En Python el operador `%` implementa esta operación y, a diferencia de algunos lenguajes, siempre devuelve un resultado no negativo cuando $n > 0$.

| $a$ | $n$ | $a \bmod n$ | Nota |
|-----|-----|-------------|------|
| 17  | 5   | 2           | $17 = 3 \cdot 5 + 2$ |
| -7  | 3   | 2           | Python: $-7 = -3 \cdot 3 + 2$ |
| 100 | 26  | 22          | |
| 0   | 7   | 0           | |


In [ ]:
# ── Ejemplos básicos de módulo ────────────────────────────────────────────────
casos = [(17, 5), (-7, 3), (100, 26), (0, 7), (2**32, 97)]

print(f"{'a':>12} {'n':>6} {'a % n':>8}")
print('-' * 30)
for a, n in casos:
    print(f"{a:>12} {n:>6} {a % n:>8}")

### 1.2 Congruencias

Decimos que $a$ es **congruente** con $b$ módulo $n$ y escribimos

$$a \equiv b \pmod{n}$$

si y sólo si $n \mid (a - b)$, es decir, $n$ divide exactamente a $a - b$.

**Ejemplo:** $17 \equiv 2 \pmod{5}$ porque $5 \mid (17-2)=15$.

Las congruencias forman una **relación de equivalencia**: la clase de equivalencia de $a$ módulo $n$ es
$$[a]_n = \{\ldots, a-2n, a-n, a, a+n, a+2n, \ldots\}$$

El conjunto de todas las clases es $\mathbb{Z}/n\mathbb{Z} = \{[0]_n, [1]_n, \ldots, [n-1]_n\}$, llamado **anillo de enteros módulo $n$**.


In [ ]:
def son_congruentes(a: int, b: int, n: int) -> bool:
    """Retorna True si a ≡ b (mod n)."""
    return (a - b) % n == 0

# Pruebas
print(son_congruentes(17, 2, 5))    # True
print(son_congruentes(100, 2, 7))   # False  (100 mod 7 = 2 → True!)
print(son_congruentes(100, 2, 7))   # True: 100 = 14*7 + 2

# Visualizar clases de equivalencia módulo 6
n = 6
rango = range(-12, 25)
colores = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(14, 3))
for x in rango:
    clase = x % n
    ax.bar(x, 1, color=colores[clase], edgecolor='black', linewidth=0.5)
    ax.text(x, 0.5, str(x), ha='center', va='center', fontsize=8, color='white', fontweight='bold')

leyenda = [plt.Rectangle((0,0),1,1, color=colores[i], label=f'Clase [{i}]') for i in range(n)]
ax.legend(handles=leyenda, loc='upper right', ncol=n)
ax.set_yticks([])
ax.set_xlabel('Enteros')
ax.set_title(f'Clases de equivalencia en $\\mathbb{{Z}}/{n}\\mathbb{{Z}}$')
plt.tight_layout()
plt.show()

---
## 2. Propiedades de la Aritmética Modular <a id='2-propiedades'></a>

La aritmética modular es **compatible** con la suma, la resta y la multiplicación:

Si $a \equiv a' \pmod{n}$ y $b \equiv b' \pmod{n}$, entonces:

| Propiedad | Fórmula |
|-----------|---------|
| Suma | $(a + b) \bmod n = (a \bmod n + b \bmod n) \bmod n$ |
| Resta | $(a - b) \bmod n = (a \bmod n - b \bmod n) \bmod n$ |
| Multiplicación | $(a \cdot b) \bmod n = ((a \bmod n) \cdot (b \bmod n)) \bmod n$ |
| Potencia | $a^k \bmod n = (a \bmod n)^k \bmod n$ |

**¡Atención!** La división modular **no siempre existe**: sólo cuando el divisor tiene inverso multiplicativo módulo $n$.


In [ ]:
# ── Demostración de propiedades ───────────────────────────────────────────────
a, b, n = 37, 58, 13

print("Suma:")
print(f"  (a + b) % n          = {(a + b) % n}")
print(f"  (a%n + b%n) % n      = {(a % n + b % n) % n}")

print("\nMultiplicación:")
print(f"  (a * b) % n          = {(a * b) % n}")
print(f"  ((a%n) * (b%n)) % n  = {((a % n) * (b % n)) % n}")

print("\nPotencia:")
k = 10
print(f"  (a**k) % n           = {(a**k) % n}")
print(f"  pow(a%n, k, n)       = {pow(a % n, k, n)}")
print(f"  pow(a, k, n)         = {pow(a, k, n)}  ← Python built-in eficiente")

In [ ]:
# ── Tabla de suma y multiplicación mod 7 ─────────────────────────────────────
m = 7
tabla_suma = np.array([[( i + j) % m for j in range(m)] for i in range(m)])
tabla_mult = np.array([[(i * j) % m for j in range(m)] for i in range(m)])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, tabla, titulo in zip(axes,
                              [tabla_suma, tabla_mult],
                              [f'Tabla de suma mod {m}', f'Tabla de multiplicación mod {m}']):
    im = ax.imshow(tabla, cmap='YlOrRd', vmin=0, vmax=m-1)
    for i in range(m):
        for j in range(m):
            ax.text(j, i, tabla[i, j], ha='center', va='center', fontsize=14, fontweight='bold')
    ax.set_xticks(range(m))
    ax.set_yticks(range(m))
    ax.set_xticklabels(range(m))
    ax.set_yticklabels(range(m))
    ax.set_title(titulo, fontsize=13)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

---
## 3. Máximo Común Divisor y Algoritmo de Euclides <a id='3-euclides'></a>

### 3.1 Definición de MCD

El **máximo común divisor** de dos enteros $a$ y $b$ (no ambos cero), $\gcd(a, b)$, es el entero positivo más grande que divide a ambos.

**Propiedades fundamentales:**
- $\gcd(a, 0) = a$
- $\gcd(a, b) = \gcd(b, a \bmod b)$  ← base del algoritmo de Euclides
- $\gcd(a, b) = \gcd(a, b + ka)$ para cualquier entero $k$
- **Identidad de Bézout:** existen enteros $x, y$ tal que $ax + by = \gcd(a, b)$

### 3.2 Algoritmo de Euclides

Publicado hacia el 300 a.C. en *Elementos* (Libro VII), es uno de los algoritmos más antiguos conocidos:

$$\gcd(a, b) = \begin{cases} a & \text{si } b = 0 \\ \gcd(b, a \bmod b) & \text{si } b \neq 0 \end{cases}$$

**Complejidad:** $O(\log(\min(a,b)))$ divisiones — exponencialmente más rápido que la factorización.


In [ ]:
# ── Algoritmo de Euclides con traza ──────────────────────────────────────────
def gcd_euclides(a: int, b: int, verbose: bool = False) -> int:
    """MCD por el algoritmo de Euclides."""
    paso = 1
    while b != 0:
        q, r = divmod(a, b)
        if verbose:
            print(f"  Paso {paso}: {a} = {q}·{b} + {r}   →  gcd({a},{b}) = gcd({b},{r})")
        a, b = b, r
        paso += 1
    if verbose:
        print(f"  gcd = {a}")
    return a

print("gcd(252, 105):")
resultado = gcd_euclides(252, 105, verbose=True)

print()
print("gcd(1071, 462):")
gcd_euclides(1071, 462, verbose=True)

# Verificación con math.gcd
print()
print("Verificación con math.gcd:", math.gcd(252, 105))

In [ ]:
# ── Visualización: número de pasos del algoritmo de Euclides ─────────────────
def pasos_euclides(a: int, b: int) -> int:
    pasos = 0
    while b:
        a, b = b, a % b
        pasos += 1
    return pasos

N = 200
mapa_pasos = np.array([[pasos_euclides(i, j) if j != 0 else 0 for j in range(N)] for i in range(N)])

plt.figure(figsize=(8, 7))
plt.imshow(mapa_pasos, cmap='inferno', origin='lower')
plt.colorbar(label='Número de pasos')
plt.title('Pasos del algoritmo de Euclides para gcd(a, b)')
plt.xlabel('b')
plt.ylabel('a')
plt.tight_layout()
plt.show()

# Los pares de Fibonacci maximizan los pasos (teorema de Lamé)
fibs = [1, 1]
while fibs[-1] < N:
    fibs.append(fibs[-1] + fibs[-2])
print("Pares de Fibonacci consecutivos (máximo pasos):")
for f1, f2 in zip(fibs[-4:], fibs[-3:]):
    print(f"  gcd({f2}, {f1}) → {pasos_euclides(f2, f1)} pasos")

### 3.3 Números Coprimos

Dos enteros $a$ y $b$ son **coprimos** (o relativamente primos) si $\gcd(a, b) = 1$. Este concepto es central en criptografía porque garantiza la existencia de inversos modulares.

**Ejemplo criptográfico:** en RSA, se requiere que $e$ y $\phi(n)$ sean coprimos para que $e$ tenga inverso modular.


In [ ]:
# ── Mapa de coprimalidad para módulo m ────────────────────────────────────────
def coprimos_con(m: int) -> List[int]:
    return [i for i in range(1, m) if math.gcd(i, m) == 1]

for m in [10, 12, 15, 17, 30]:
    cops = coprimos_con(m)
    print(f"Coprimos con {m:2d}: {cops}  (φ({m}) = {len(cops)})")

---
## 4. Algoritmo de Euclides Extendido <a id='4-euclides-ext'></a>

El **algoritmo de Euclides extendido** no sólo calcula $\gcd(a, b)$, sino que también encuentra los coeficientes $x, y$ enteros que satisfacen la **identidad de Bézout**:

$$ax + by = \gcd(a, b)$$

### Algoritmo (versión iterativa)

Mantenemos tres secuencias paralelas:

$$r_0 = a,\; r_1 = b, \quad s_0 = 1,\; s_1 = 0, \quad t_0 = 0,\; t_1 = 1$$

En cada iteración: $q_i = \lfloor r_{i-1}/r_i \rfloor$ y
$$r_{i+1} = r_{i-1} - q_i \cdot r_i, \quad s_{i+1} = s_{i-1} - q_i \cdot s_i, \quad t_{i+1} = t_{i-1} - q_i \cdot t_i$$

El algoritmo termina cuando $r_{k+1} = 0$. Entonces $\gcd(a,b) = r_k$, $x = s_k$, $y = t_k$.


In [ ]:
# ── Algoritmo de Euclides Extendido ──────────────────────────────────────────
def euclides_extendido(a: int, b: int, verbose: bool = False) -> Tuple[int, int, int]:
    """
    Retorna (g, x, y) tal que a*x + b*y = g = gcd(a, b).
    """
    r_prev, r_curr = a, b
    s_prev, s_curr = 1, 0
    t_prev, t_curr = 0, 1

    if verbose:
        print(f"{'r':>10} {'s':>10} {'t':>10} {'q':>10}")
        print(f"{r_prev:>10} {s_prev:>10} {t_prev:>10} {'':>10}")
        print(f"{r_curr:>10} {s_curr:>10} {t_curr:>10} {'':>10}")

    while r_curr != 0:
        q = r_prev // r_curr
        r_prev, r_curr = r_curr, r_prev - q * r_curr
        s_prev, s_curr = s_curr, s_prev - q * s_curr
        t_prev, t_curr = t_curr, t_prev - q * t_curr
        if verbose:
            print(f"{r_prev:>10} {s_prev:>10} {t_prev:>10} {q:>10}")

    return r_prev, s_prev, t_prev


# Ejemplo 1
a, b = 252, 105
g, x, y = euclides_extendido(a, b, verbose=True)
print(f"\ngcd({a}, {b}) = {g}")
print(f"{a}·({x}) + {b}·({y}) = {a*x + b*y}  ✓" if a*x + b*y == g else "ERROR")

print()
# Ejemplo 2 — coprimos
a, b = 35, 27
g, x, y = euclides_extendido(a, b)
print(f"gcd({a}, {b}) = {g}")
print(f"{a}·({x}) + {b}·({y}) = {a*x + b*y}")

> **Importancia criptográfica:** La identidad de Bézout es la base del cálculo de inversos modulares. Si $\gcd(a, n) = 1$, entonces $ax + ny = 1$ implica $ax \equiv 1 \pmod{n}$, es decir, $x$ es el inverso de $a$ módulo $n$.


---
## 5. Inversos Modulares <a id='5-inversos'></a>

### 5.1 Definición

El **inverso multiplicativo** de $a$ módulo $n$ es el entero $a^{-1}$ tal que:

$$a \cdot a^{-1} \equiv 1 \pmod{n}$$

**Existencia:** $a^{-1} \bmod n$ existe **si y sólo si** $\gcd(a, n) = 1$.

### 5.2 Cálculo

**Método 1 — Euclides extendido:** Si $ax + ny = 1$, entonces $a^{-1} \equiv x \pmod{n}$.

**Método 2 — Pequeño Teorema de Fermat** (sólo cuando $n$ es primo): $a^{-1} \equiv a^{n-2} \pmod{n}$.

**Método 3 — Python built-in:** `pow(a, -1, n)` (Python ≥ 3.8).


In [ ]:
# ── Cálculo de inverso modular ────────────────────────────────────────────────
def inverso_modular(a: int, n: int) -> int:
    """
    Retorna a^{-1} mod n usando el algoritmo de Euclides extendido.
    Lanza ValueError si no existe.
    """
    g, x, _ = euclides_extendido(a, n)
    if g != 1:
        raise ValueError(f"El inverso de {a} mod {n} no existe (gcd={g}).")
    return x % n


# Ejemplos
pares = [(3, 7), (7, 26), (17, 3120), (65537, 10^20 + 7)]

for a, n in pares:
    try:
        inv = inverso_modular(a, n)
        verificacion = (a * inv) % n
        print(f"  {a}^(-1) mod {n} = {inv}   [verificación: {a}·{inv} ≡ {verificacion} (mod {n})]")
    except ValueError as e:
        print(f"  {e}")

print()
# Caso sin inverso
try:
    inverso_modular(6, 9)
except ValueError as e:
    print(f"  Esperado: {e}")

# Comparación con pow() built-in de Python
print()
a, n = 7, 26
print(f"  pow(7, -1, 26) = {pow(7, -1, 26)}  (Python 3.8+)")

In [ ]:
# ── Tabla de inversos módulo 11 (primo) ───────────────────────────────────────
n = 11
print(f"Tabla de inversos módulo {n}:")
print(f"{'a':>5} {'a^-1 mod n':>12} {'Verificación a·a^-1 mod n':>26}")
print('-' * 46)
for a in range(1, n):
    inv = inverso_modular(a, n)
    print(f"{a:>5} {inv:>12} {(a * inv) % n:>26}")

### 5.3 Aplicación: Cifrado de Hill

El **cifrado de Hill** (1929) usa matrices para cifrar bloques. El descifrado requiere encontrar la **matriz inversa** modular, que existe sólo si el determinante de la matriz tiene inverso módulo 26.


In [ ]:
# ── Cifrado de Hill 2×2 ───────────────────────────────────────────────────────
def hill_cifrar(texto: str, K: np.ndarray, mod: int = 26) -> str:
    texto = texto.upper().replace(' ', '')
    if len(texto) % 2 != 0:
        texto += 'X'
    cifrado = []
    for i in range(0, len(texto), 2):
        bloque = np.array([[ord(texto[i]) - 65], [ord(texto[i+1]) - 65]])
        resultado = (K @ bloque) % mod
        cifrado.append(chr(int(resultado[0]) + 65))
        cifrado.append(chr(int(resultado[1]) + 65))
    return ''.join(cifrado)


def matriz_inversa_mod(K: np.ndarray, mod: int = 26) -> np.ndarray:
    """Inversa modular de una matriz 2x2."""
    det = int(round(np.linalg.det(K)))
    det_inv = inverso_modular(det % mod, mod)
    adj = np.array([[ K[1,1], -K[0,1]],
                    [-K[1,0],  K[0,0]]])
    return (det_inv * adj % mod).astype(int)


def hill_descifrar(cifrado: str, K: np.ndarray, mod: int = 26) -> str:
    K_inv = matriz_inversa_mod(K, mod)
    return hill_cifrar(cifrado, K_inv, mod)


# Clave
K = np.array([[3, 3],
              [2, 5]])

det = int(round(np.linalg.det(K))) % 26
print(f"Clave K:\n{K}")
print(f"det(K) mod 26 = {det}, gcd(det, 26) = {math.gcd(det, 26)}")
print()

mensaje = "HOLA"
cifrado  = hill_cifrar(mensaje, K)
descifrado = hill_descifrar(cifrado, K)

print(f"Mensaje   : {mensaje}")
print(f"Cifrado   : {cifrado}")
print(f"Descifrado: {descifrado}")

---
## 6. Exponenciación Modular Rápida <a id='6-expmod'></a>

### 6.1 El problema

En criptografía es común calcular $a^e \bmod n$ con exponentes **enormes** (ej. $e = 65537$ o $e$ de 2048 bits). La exponenciación naive requiere $e-1$ multiplicaciones; la **exponenciación rápida** (*square-and-multiply*) sólo necesita $O(\log e)$ multiplicaciones.

### 6.2 Algoritmo Square-and-Multiply

Se basa en la representación binaria del exponente $e = (b_k b_{k-1} \ldots b_1 b_0)_2$:

```
resultado = 1
base = a mod n
mientras e > 0:
    si e es impar: resultado = resultado * base mod n
    base = base² mod n
    e = e >> 1   (división entera entre 2)
```


In [ ]:
# ── Exponenciación modular rápida ─────────────────────────────────────────────
def exp_mod_rapida(base: int, exp: int, mod: int, verbose: bool = False) -> int:
    """
    Calcula base^exp mod mod usando square-and-multiply.
    Equivalente a pow(base, exp, mod) de Python.
    """
    resultado = 1
    base = base % mod
    bits = bin(exp)[2:]  # representación binaria sin '0b'

    if verbose:
        print(f"  Exponente {exp} en binario: {bits}")
        print(f"  {'Bit':>5} {'Base':>20} {'Resultado':>20}")

    for bit in bits:
        resultado = (resultado * resultado) % mod  # siempre elevar al cuadrado
        if bit == '1':
            resultado = (resultado * base) % mod   # multiplicar si bit = 1
        if verbose:
            print(f"  {bit:>5} {base:>20} {resultado:>20}")

    return resultado


# Demostración
base, exp, mod = 7, 644, 645
print(f"Calculando {base}^{exp} mod {mod}:")
res = exp_mod_rapida(base, exp, mod, verbose=True)
print(f"  Resultado: {res}")
print(f"  Verificar con pow(): {pow(base, exp, mod)}")

In [ ]:
# ── Comparación de rendimiento: naive vs. rápida ──────────────────────────────
import time

def exp_mod_naive(base: int, exp: int, mod: int) -> int:
    resultado = 1
    for _ in range(exp):
        resultado = (resultado * base) % mod
    return resultado

base, mod = 3, 10**9 + 7
exponentes = [100, 1_000, 10_000, 100_000]

print(f"{'Exponente':>12} {'Naive (ms)':>12} {'Rápida (ms)':>13} {'Speedup':>10}")
print('-' * 52)

for exp in exponentes:
    t0 = time.perf_counter()
    r1 = exp_mod_naive(base, exp, mod)
    t_naive = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    r2 = pow(base, exp, mod)
    t_rapida = (time.perf_counter() - t0) * 1000

    speedup = t_naive / t_rapida if t_rapida > 0 else float('inf')
    print(f"{exp:>12,} {t_naive:>12.3f} {t_rapida:>13.4f} {speedup:>10.0f}x")
    assert r1 == r2, "¡Los resultados difieren!"

---
## 7. Teorema Chino del Resto <a id='7-crt'></a>

### 7.1 Enunciado

**Teorema (Sun Tzu, ~3er siglo d.C.):** Sean $n_1, n_2, \ldots, n_k$ enteros positivos **coprimos dos a dos** (es decir, $\gcd(n_i, n_j) = 1$ para $i \neq j$). Entonces para cualquier $a_1, \ldots, a_k$, el sistema

$$x \equiv a_1 \pmod{n_1}, \quad x \equiv a_2 \pmod{n_2}, \quad \ldots, \quad x \equiv a_k \pmod{n_k}$$

tiene **una única solución** módulo $N = n_1 \cdot n_2 \cdots n_k$.

### 7.2 Construcción de la solución

Sea $N = \prod n_i$. Para cada $i$:
1. $N_i = N / n_i$
2. $M_i = N_i^{-1} \bmod n_i$ (inverso modular)
3. Solución: $x \equiv \sum_{i} a_i \cdot N_i \cdot M_i \pmod{N}$


In [ ]:
# ── Teorema Chino del Resto ───────────────────────────────────────────────────
def crt(residuos: List[int], modulos: List[int], verbose: bool = False) -> int:
    """
    Resuelve el sistema x ≡ residuos[i] (mod modulos[i]).
    Requiere que los módulos sean coprimos dos a dos.
    Retorna la solución mínima no negativa.
    """
    # Verificar coprimalidad
    for i in range(len(modulos)):
        for j in range(i+1, len(modulos)):
            if math.gcd(modulos[i], modulos[j]) != 1:
                raise ValueError(f"Los módulos {modulos[i]} y {modulos[j]} no son coprimos.")

    N = 1
    for m in modulos:
        N *= m

    x = 0
    for a_i, n_i in zip(residuos, modulos):
        N_i = N // n_i
        M_i = inverso_modular(N_i, n_i)
        contribucion = a_i * N_i * M_i
        x += contribucion
        if verbose:
            print(f"  a_i={a_i}, n_i={n_i}, N_i={N_i}, M_i={M_i}, contribución={contribucion}")

    return x % N


# Ejemplo clásico: Sun Tzu
# x ≡ 2 (mod 3), x ≡ 3 (mod 5), x ≡ 2 (mod 7)
residuos = [2, 3, 2]
modulos  = [3, 5, 7]
print("Sistema de congruencias:")
for a, n in zip(residuos, modulos):
    print(f"  x ≡ {a} (mod {n})")

x = crt(residuos, modulos, verbose=True)
N = 3 * 5 * 7
print(f"\nSolución: x ≡ {x} (mod {N})")
print("Verificación:", all((x % n) == a for a, n in zip(residuos, modulos)))

In [ ]:
# ── Aplicación: CRT para acelerar RSA ─────────────────────────────────────────
# En RSA, si se conocen p y q, se puede calcular m = c^d mod n
# de forma más rápida usando CRT (optimización de Garner):
#
#   m_p = c^(d mod p-1) mod p
#   m_q = c^(d mod q-1) mod q
#   Luego combinar con CRT.

def rsa_crt_descifrar(c: int, d: int, p: int, q: int) -> int:
    """Descifrado RSA acelerado con CRT."""
    dp = d % (p - 1)
    dq = d % (q - 1)
    m_p = pow(c, dp, p)
    m_q = pow(c, dq, q)
    return crt([m_p, m_q], [p, q])

# Ejemplo numérico pequeño
p, q = 61, 53
n = p * q           # 3233
e = 17
phi = (p-1)*(q-1)  # 3120
d = inverso_modular(e, phi)

mensaje = 65
cifrado = pow(mensaje, e, n)
desc_normal = pow(cifrado, d, n)
desc_crt    = rsa_crt_descifrar(cifrado, d, p, q)

print(f"n={n}, e={e}, d={d}")
print(f"Mensaje original : {mensaje}")
print(f"Cifrado          : {cifrado}")
print(f"Descifrado normal: {desc_normal}")
print(f"Descifrado CRT   : {desc_crt}")
print(f"Coinciden: {desc_normal == desc_crt}")

In [ ]:
# ── Visualización del CRT: solución en el cuadrado de residuos ────────────────
n1, n2 = 5, 7
N = n1 * n2

# Para cada x en [0, N), calcular (x mod n1, x mod n2)
xs = list(range(N))
r1s = [x % n1 for x in xs]
r2s = [x % n2 for x in xs]

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(r1s, r2s, c=xs, cmap='rainbow', s=200, zorder=5, edgecolors='black')
for x, r1, r2 in zip(xs, r1s, r2s):
    ax.annotate(str(x), (r1, r2), textcoords='offset points', xytext=(7, 4), fontsize=9)

plt.colorbar(scatter, label='Valor de x mod 35')
ax.set_xticks(range(n1))
ax.set_yticks(range(n2))
ax.set_xlabel(f'x mod {n1}')
ax.set_ylabel(f'x mod {n2}')
ax.set_title(f'CRT: biyección entre $\\mathbb{{Z}}/{N}\\mathbb{{Z}}$ y $\\mathbb{{Z}}/{n1}\\mathbb{{Z}} \\times \\mathbb{{Z}}/{n2}\\mathbb{{Z}}$')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Cada punto (x mod 5, x mod 7) es único → biyección del CRT")

---
## 8. Pequeño Teorema de Fermat <a id='8-fermat'></a>

### 8.1 Enunciado

**Teorema (Pierre de Fermat, 1640):** Si $p$ es primo y $\gcd(a, p) = 1$, entonces:

$$a^{p-1} \equiv 1 \pmod{p}$$

**Corolario:** Para cualquier entero $a$ y primo $p$:
$$a^p \equiv a \pmod{p}$$

### 8.2 Consecuencias inmediatas

1. **Inverso modular:** $a^{-1} \equiv a^{p-2} \pmod{p}$ (cuando $p$ primo).
2. **Reducción de exponentes:** $a^e \equiv a^{e \bmod (p-1)} \pmod{p}$ (para $\gcd(a,p)=1$).
3. **Test de primalidad probabilístico** (ver Sección 10).


In [ ]:
# ── Verificación del PTF y consecuencias ─────────────────────────────────────
def verificar_ptf(p: int, num_pruebas: int = 10) -> bool:
    """Verifica el PTF para p con num_pruebas bases aleatorias."""
    for _ in range(num_pruebas):
        a = random.randint(2, p - 1)
        if pow(a, p - 1, p) != 1:
            return False
    return True

primos = [5, 7, 11, 13, 17, 23, 29, 31, 97, 101]
compuestos = [4, 6, 9, 15, 25, 35, 49, 91]

print("Verificación del Pequeño Teorema de Fermat:")
print(f"  {'p':>6} {'¿Primo?':>8} {'PTF pasa':>10}")
print('  ' + '-' * 28)
for n in primos + compuestos:
    es_primo = sympy.isprime(n)
    ptf = verificar_ptf(n)
    print(f"  {n:>6} {str(es_primo):>8} {str(ptf):>10}")

In [ ]:
# ── Números de Carmichael: compuestos que pasan el test de Fermat ─────────────
# Un número de Carmichael es compuesto pero satisface a^(n-1) ≡ 1 (mod n)
# para todo a coprimo con n. Son la razón por la que el test de Fermat
# puro es inadecuado para producción.

def es_carmichael(n: int) -> bool:
    if sympy.isprime(n) or n < 2:
        return False
    # a^(n-1) ≡ 1 (mod n) para todo a coprimo con n
    return all(pow(a, n-1, n) == 1 for a in range(2, min(n, 1000)) if math.gcd(a, n) == 1)

# Los primeros números de Carmichael
carmichaels = [n for n in range(2, 10000) if es_carmichael(n)]
print("Primeros números de Carmichael (< 10 000):")
print(carmichaels)
print(f"\nTotal encontrados: {len(carmichaels)}")
print("\nVer que 561 = 3 × 11 × 17 pasa el test de Fermat:")
n = 561
for a in [2, 5, 7, 13]:
    print(f"  {a}^560 mod 561 = {pow(a, n-1, n)}")

> **Nota de seguridad:** Los números de Carmichael muestran que el test de Fermat simple **no es suficiente** para verificar primalidad en criptografía. Se usa en cambio el test de Miller-Rabin (Sección 10).


---
## 9. Función Phi de Euler y Teorema de Euler <a id='9-euler'></a>

### 9.1 Función Totiente de Euler

La **función phi de Euler** $\phi(n)$ cuenta cuántos enteros en $\{1, 2, \ldots, n\}$ son coprimos con $n$:

$$\phi(n) = |\{k : 1 \le k \le n, \gcd(k,n) = 1\}|$$

**Fórmula para la factorización $n = p_1^{e_1} \cdots p_r^{e_r}$:**

$$\phi(n) = n \prod_{p \mid n} \left(1 - \frac{1}{p}\right)$$

**Casos importantes:**

| $n$ | $\phi(n)$ | Observación |
|-----|-----------|-------------|
| $p$ (primo) | $p-1$ | Todos los enteros menores son coprimos |
| $p^k$ | $p^k - p^{k-1} = p^{k-1}(p-1)$ | |
| $pq$ (primos distintos) | $(p-1)(q-1)$ | ← Usado en RSA |
| $n = 1$ | $1$ | |


In [ ]:
# ── Función phi de Euler ──────────────────────────────────────────────────────
def phi_euler(n: int) -> int:
    """Calcula φ(n) usando la fórmula del producto."""
    resultado = n
    p = 2
    temp = n
    while p * p <= temp:
        if temp % p == 0:
            while temp % p == 0:
                temp //= p
            resultado -= resultado // p
        p += 1
    if temp > 1:
        resultado -= resultado // temp
    return resultado


# Tabla de phi
print(f"{'n':>5} {'φ(n)':>8} {'φ(n)/n':>10} {'Coprimos con n'}")
print('-' * 55)
for n in range(1, 21):
    phi = phi_euler(n)
    cops = [k for k in range(1, n+1) if math.gcd(k, n) == 1]
    print(f"{n:>5} {phi:>8} {phi/n:>10.4f}  {cops}")

In [ ]:
# ── Visualización de φ(n) ─────────────────────────────────────────────────────
ns = list(range(1, 201))
phis = [phi_euler(n) for n in ns]
ratios = [p/n for p, n in zip(phis, ns)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(ns, phis, s=10, alpha=0.7, c=phis, cmap='viridis')
axes[0].plot(ns, [n-1 for n in ns], 'r--', alpha=0.4, label='n-1 (primo)')
axes[0].set_xlabel('n')
axes[0].set_ylabel('φ(n)')
axes[0].set_title('Función Totiente de Euler φ(n)')
axes[0].legend()

axes[1].scatter(ns, ratios, s=10, alpha=0.7, c=ratios, cmap='plasma')
axes[1].set_xlabel('n')
axes[1].set_ylabel('φ(n)/n')
axes[1].set_title('Densidad de coprimos φ(n)/n')
axes[1].axhline(y=1, color='r', linestyle='--', alpha=0.4, label='Primos')
axes[1].legend()

plt.tight_layout()
plt.show()

### 9.2 Teorema de Euler

**Teorema (Leonhard Euler, 1763):** Si $\gcd(a, n) = 1$, entonces:

$$a^{\phi(n)} \equiv 1 \pmod{n}$$

El Pequeño Teorema de Fermat es el caso especial $n = p$ (primo), donde $\phi(p) = p - 1$.

### 9.3 Orden multiplicativo

El **orden** de $a$ módulo $n$, $\text{ord}_n(a)$, es el menor entero positivo $k$ tal que $a^k \equiv 1 \pmod{n}$. El teorema de Euler garantiza que $\text{ord}_n(a)$ divide a $\phi(n)$.


In [ ]:
# ── Orden multiplicativo y generadores ───────────────────────────────────────
def orden_multiplicativo(a: int, n: int) -> int:
    """Calcula el orden de a en (Z/nZ)*."""
    if math.gcd(a, n) != 1:
        raise ValueError(f"{a} no es invertible mod {n}")
    k, r = 1, a % n
    while r != 1:
        r = (r * a) % n
        k += 1
    return k


p = 13
phi_p = p - 1
print(f"Órdenes en (Z/{p}Z)*  [φ({p}) = {phi_p}]:")
print(f"  {'a':>4} {'ord(a)':>8} {'¿Generador?':>13} {'Potencias'}")
print('  ' + '-' * 65)

for a in range(1, p):
    ord_a = orden_multiplicativo(a, p)
    potencias = [pow(a, k, p) for k in range(1, ord_a + 1)]
    gen = '✓' if ord_a == phi_p else ''
    print(f"  {a:>4} {ord_a:>8} {gen:>13} {potencias}")

In [ ]:
# ── Raíces primitivas (generadores) ──────────────────────────────────────────
def raices_primitivas(p: int) -> List[int]:
    """Retorna todas las raíces primitivas del primo p."""
    return [a for a in range(1, p) if orden_multiplicativo(a, p) == p - 1]

for p in [7, 11, 13, 17, 19, 23]:
    rp = raices_primitivas(p)
    print(f"p={p:3d}: {len(rp):2d} raíces primitivas → {rp}")

---
## 10. Números Primos y Tests de Primalidad <a id='10-primalidad'></a>

### 10.1 Importancia en criptografía

Los números primos grandes son esenciales en RSA, Diffie-Hellman y curvas elípticas. Necesitamos:
1. **Generar** primos grandes eficientemente.
2. **Verificar** que un candidato es primo de forma determinista o probabilística.

### 10.2 Test de Miller-Rabin

Se basa en que si $p$ es primo impar, para cualquier $a$ con $\gcd(a,p)=1$, si escribimos $p-1 = 2^s \cdot d$ (con $d$ impar), entonces:

$$a^d \equiv 1 \pmod{p} \quad \text{o bien} \quad a^{2^r d} \equiv -1 \pmod{p} \text{ para algún } 0 \le r < s$$

Un número que **no** satisface ninguna de estas condiciones para alguna base $a$ es **compuesto con certeza**. Con $k$ bases aleatorias, la probabilidad de error es $\le 4^{-k}$.


In [ ]:
# ── Test de Miller-Rabin ──────────────────────────────────────────────────────
def miller_rabin(n: int, k: int = 40) -> bool:
    """
    Test de primalidad probabilístico Miller-Rabin.
    Retorna True si n es probablemente primo, False si es compuesto.
    Con k=40, la probabilidad de error es < 4^{-40} ≈ 10^{-24}.
    """
    if n < 2:
        return False
    if n == 2 or n == 3:
        return True
    if n % 2 == 0:
        return False

    # Escribir n-1 = 2^s * d con d impar
    s, d = 0, n - 1
    while d % 2 == 0:
        s += 1
        d //= 2

    # k rondas con bases aleatorias
    for _ in range(k):
        a = random.randrange(2, n - 1)
        x = pow(a, d, n)

        if x == 1 or x == n - 1:
            continue

        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False  # compuesto

    return True  # probablemente primo


# Comparar con sympy
import time
candidatos = [2, 3, 4, 17, 100, 561, 6700417, 2**31 - 1, 2**61 - 1, 2**127 - 1]

print(f"{'n':>25} {'Miller-Rabin':>14} {'sympy.isprime':>14} {'Coinciden':>10}")
print('-' * 68)
for n in candidatos:
    mr = miller_rabin(n)
    sp = sympy.isprime(n)
    ok = '✓' if mr == sp else '✗ ERROR'
    print(f"{n:>25} {str(mr):>14} {str(sp):>14} {ok:>10}")

In [ ]:
# ── Generación de primos grandes ─────────────────────────────────────────────
def generar_primo(bits: int) -> int:
    """
    Genera un número primo aleatorio de 'bits' bits
    usando el test de Miller-Rabin.
    """
    while True:
        # Candidato: número impar aleatorio de 'bits' bits
        candidato = random.getrandbits(bits)
        # Asegurar que tenga exactamente 'bits' bits (bit más significativo = 1)
        candidato |= (1 << (bits - 1))
        # Asegurar que sea impar
        candidato |= 1
        if miller_rabin(candidato):
            return candidato

# Generar primos de distintos tamaños
for bits in [16, 32, 64, 128, 256, 512]:
    p = generar_primo(bits)
    print(f"  {bits:4d} bits: {p}")
    assert miller_rabin(p), "Error: el número generado no es primo"

In [ ]:
# ── Criba de Eratóstenes ──────────────────────────────────────────────────────
def criba_eratostenes(limite: int) -> List[int]:
    """Retorna todos los primos hasta 'limite' inclusive."""
    es_primo = bytearray([1]) * (limite + 1)
    es_primo[0] = es_primo[1] = 0
    for i in range(2, int(limite**0.5) + 1):
        if es_primo[i]:
            es_primo[i*i::i] = bytearray(len(es_primo[i*i::i]))
    return [i for i, p in enumerate(es_primo) if p]

primos_100 = criba_eratostenes(100)
print(f"Primos hasta 100 ({len(primos_100)} primos):")
print(primos_100)

# Teorema del número primo: π(x) ≈ x / ln(x)
xs = range(10, 100001, 1000)
pi_x  = [len(criba_eratostenes(x)) for x in xs]
aprox = [x / math.log(x) for x in xs]

plt.figure(figsize=(12, 5))
plt.plot(xs, pi_x, label=r'$\pi(x)$ (conteo exacto)', linewidth=2)
plt.plot(xs, aprox, '--', label=r'$x / \ln(x)$ (aproximación)', linewidth=2)
plt.xlabel('x')
plt.ylabel('Número de primos')
plt.title('Teorema del Número Primo: $\\pi(x) \\approx x/\\ln(x)$')
plt.legend()
plt.tight_layout()
plt.show()

---
## 11. Aplicación: RSA desde cero <a id='11-rsa'></a>

### 11.1 Historia y contexto

El algoritmo RSA fue publicado en 1977 por **Rivest, Shamir y Adleman** (MIT). Es el criptosistema de clave pública más utilizado en el mundo, presente en TLS/HTTPS, SSH, PGP, y documentos digitales.

Su seguridad descansa en la **dificultad de factorizar** el producto de dos primos grandes.

### 11.2 Algoritmo completo

**Generación de claves:**
1. Elegir dos primos distintos grandes $p$ y $q$.
2. Calcular $n = p \cdot q$ (módulo público).
3. Calcular $\phi(n) = (p-1)(q-1)$.
4. Elegir $e$ tal que $1 < e < \phi(n)$ y $\gcd(e, \phi(n)) = 1$ (exponente público).
5. Calcular $d = e^{-1} \bmod \phi(n)$ (clave privada).
6. Clave pública: $(n, e)$. Clave privada: $(n, d)$.

**Cifrado:** $c = m^e \bmod n$ (requiere $0 \le m < n$).

**Descifrado:** $m = c^d \bmod n$.

**Correctitud:** Del teorema de Euler, $m^{e \cdot d} = m^{1 + k\phi(n)} \equiv m \pmod{n}$.


In [ ]:
# ── RSA completo desde cero ───────────────────────────────────────────────────
class RSA:
    def __init__(self, bits: int = 512):
        """Genera un par de claves RSA de 'bits' bits (bits del módulo n)."""
        mitad = bits // 2
        # Generar p y q distintos
        self.p = generar_primo(mitad)
        self.q = generar_primo(mitad)
        while self.q == self.p:
            self.q = generar_primo(mitad)

        self.n = self.p * self.q
        self.phi_n = (self.p - 1) * (self.q - 1)

        # Exponente público estándar
        self.e = 65537
        # En la práctica siempre se verifica gcd(e, phi) = 1
        assert math.gcd(self.e, self.phi_n) == 1, "Elegir otro e"

        self.d = inverso_modular(self.e, self.phi_n)

    @property
    def clave_publica(self) -> Tuple[int, int]:
        return (self.n, self.e)

    @property
    def clave_privada(self) -> Tuple[int, int]:
        return (self.n, self.d)

    def cifrar(self, m: int) -> int:
        """Cifra el entero m (0 ≤ m < n)."""
        return pow(m, self.e, self.n)

    def descifrar(self, c: int) -> int:
        """Descifra el criptotexto c."""
        return pow(c, self.d, self.n)

    def cifrar_texto(self, texto: str) -> List[int]:
        """Cifra un string byte a byte."""
        return [self.cifrar(b) for b in texto.encode('utf-8')]

    def descifrar_texto(self, cifrado: List[int]) -> str:
        """Descifra una lista de enteros a string."""
        return bytes([self.descifrar(c) for c in cifrado]).decode('utf-8')

    def info(self):
        bits_n = self.n.bit_length()
        print(f"=== Claves RSA ({bits_n} bits) ===")
        print(f"  p       = {self.p}")
        print(f"  q       = {self.q}")
        print(f"  n       = {self.n}")
        print(f"  φ(n)    = {self.phi_n}")
        print(f"  e       = {self.e}")
        print(f"  d       = {self.d}")
        print(f"  Clave pública  : (n, e)")
        print(f"  Clave privada  : (n, d)")


# Demostración con claves pequeñas (para visualización)
rsa_demo = RSA(bits=64)
rsa_demo.info()

print()
mensaje = 42
cifrado = rsa_demo.cifrar(mensaje)
descifrado = rsa_demo.descifrar(cifrado)
print(f"Mensaje     : {mensaje}")
print(f"Cifrado     : {cifrado}")
print(f"Descifrado  : {descifrado}")
print(f"Correcto    : {mensaje == descifrado}")

In [ ]:
# ── Cifrado de texto con RSA ──────────────────────────────────────────────────
rsa = RSA(bits=128)
texto_original = "Hola, mundo!"

bloques_cifrados = rsa.cifrar_texto(texto_original)
texto_recuperado = rsa.descifrar_texto(bloques_cifrados)

print(f"Texto original  : {texto_original}")
print(f"Bloques cifrados: {bloques_cifrados[:5]}... (total: {len(bloques_cifrados)})")
print(f"Texto recuperado: {texto_recuperado}")
print(f"Integridad      : {texto_original == texto_recuperado}")

In [ ]:
# ── Vulnerabilidad: factorización por fuerza bruta para n pequeño ─────────────
def factorizar_n(n: int) -> Tuple[int, int]:
    """Factoriza n = p*q por fuerza bruta (solo para n pequeño)."""
    for p in range(2, int(n**0.5) + 1):
        if n % p == 0:
            return p, n // p
    raise ValueError("No se pudo factorizar")


# Ataque a RSA con módulo pequeño (64 bits → vulnerable)
rsa_vulnerable = RSA(bits=32)
n, e = rsa_vulnerable.clave_publica
print(f"Módulo público n = {n} ({n.bit_length()} bits)")

import time
t0 = time.perf_counter()
p_rec, q_rec = factorizar_n(n)
t_factor = time.perf_counter() - t0

phi_rec = (p_rec - 1) * (q_rec - 1)
d_rec = inverso_modular(e, phi_rec)

print(f"p recuperado = {p_rec}, q recuperado = {q_rec}")
print(f"d recuperado = {d_rec}")
print(f"d correcto   = {rsa_vulnerable.d}")
print(f"Tiempo de factorización: {t_factor*1000:.2f} ms")
print()
print("→ Por eso RSA usa módulos de 2048 o 4096 bits en producción.")

---
## 12. Ejercicios y Desafíos <a id='12-ejercicios'></a>

### Nivel básico

1. Calcule $\gcd(2024, 748)$ usando el algoritmo de Euclides. Verifique con `math.gcd`.
2. Encuentre los coeficientes de Bézout $x, y$ tal que $2024x + 748y = \gcd(2024, 748)$.
3. Calcule el inverso de $17$ módulo $3120$ usando el algoritmo de Euclides extendido.
4. Resuelva el sistema: $x \equiv 1 \pmod{3}$, $x \equiv 4 \pmod{5}$, $x \equiv 6 \pmod{7}$.

### Nivel intermedio

5. Verifique el Pequeño Teorema de Fermat para $a=2, 3, 5$ y $p=997$ (primo).
6. Calcule $\phi(360)$ y liste todos los enteros coprimos con 360 en $[1, 360]$.
7. Encuentre todas las raíces primitivas módulo $23$.
8. Implemente un test de primalidad determinista basado en bases fijas de Miller-Rabin para $n < 3.3 \times 10^{24}$.

### Nivel avanzado

9. **Ataque de factor común:** Se sabe que dos módulos RSA $n_1$ y $n_2$ comparten un factor primo. Explote esto para recuperar las claves privadas de ambos.
10. **RSA textbook vs. OAEP:** Investigue por qué el RSA "libro de texto" es inseguro en la práctica. Implemente el cifrado de un mismo mensaje dos veces y compare.
11. **Logaritmo discreto:** Dado $g=2$, $p=23$, y $h = g^x \bmod p$, encuentre $x$ por fuerza bruta (algoritmo baby-step giant-step).


In [ ]:
# ── Solución: Ejercicio 9 — Ataque de factor común ───────────────────────────
def ataque_factor_comun(n1: int, e1: int, n2: int, e2: int) -> Tuple[int, int]:
    """
    Si gcd(n1, n2) > 1, ambos módulos comparten un primo.
    Retorna (d1, d2): las claves privadas recuperadas.
    """
    p = math.gcd(n1, n2)
    if p in (1, n1, n2):
        raise ValueError("Los módulos no comparten factor")

    q1 = n1 // p
    q2 = n2 // p
    phi1 = (p - 1) * (q1 - 1)
    phi2 = (p - 1) * (q2 - 1)
    d1 = inverso_modular(e1, phi1)
    d2 = inverso_modular(e2, phi2)

    print(f"  Factor común p = {p}")
    print(f"  q1 = {q1}, q2 = {q2}")
    print(f"  d1 = {d1}, d2 = {d2}")
    return d1, d2


# Crear dos claves RSA débiles que comparten primo p (escenario realista
# si el generador de números aleatorios tiene entropía insuficiente)
p_comun = generar_primo(32)
q1 = generar_primo(32)
q2 = generar_primo(32)
n1 = p_comun * q1
n2 = p_comun * q2
e_pub = 65537

print("=== Ataque de Factor Común ===")
print(f"n1 = {n1}")
print(f"n2 = {n2}")
ataque_factor_comun(n1, e_pub, n2, e_pub)

In [ ]:
# ── Solución: Ejercicio 11 — Logaritmo discreto (Baby-step Giant-step) ────────
def baby_step_giant_step(g: int, h: int, p: int) -> int:
    """
    Resuelve h = g^x mod p para x (logaritmo discreto).
    Complejidad: O(sqrt(p)) tiempo y espacio.
    """
    m = math.isqrt(p) + 1

    # Baby steps: tabla {g^j mod p : j} para j = 0..m-1
    tabla = {pow(g, j, p): j for j in range(m)}

    # Giant steps: buscar h * (g^{-m})^i mod p en tabla
    g_inv_m = pow(g, -m, p)  # g^{-m} mod p
    gamma = h
    for i in range(m):
        if gamma in tabla:
            x = i * m + tabla[gamma]
            return x % (p - 1)
        gamma = (gamma * g_inv_m) % p

    raise ValueError("No se encontró logaritmo discreto")


# Pruebas
p = 23
g = 5  # raíz primitiva mod 23

print(f"Logaritmos discretos base g={g} mod p={p}:")
print(f"{'x':>5} {'h = g^x':>10} {'log_g(h) recuperado':>22} {'Correcto':>10}")
print('-' * 52)
for x in range(1, p):
    h = pow(g, x, p)
    x_rec = baby_step_giant_step(g, h, p)
    ok = pow(g, x_rec, p) == h
    print(f"{x:>5} {h:>10} {x_rec:>22} {str(ok):>10}")

---
## 13. Referencias <a id='13-referencias'></a>

### Libros de texto

1. **Stinson, D. R., & Paterson, M.** (2018). *Cryptography: Theory and Practice* (4th ed.). CRC Press. — Capítulos 5 y 6 cubren aritmética modular y RSA en detalle.

2. **Boneh, D., & Shoup, V.** (2023). *A Graduate Course in Applied Cryptography*. Disponible en: https://toc.cryptobook.us — Capítulos 11–13 sobre criptografía de clave pública.

3. **Hoffstein, J., Pipher, J., & Silverman, J. H.** (2014). *An Introduction to Mathematical Cryptography* (2nd ed.). Springer. — Capítulos 1 y 3 sobre aritmética modular y RSA.

4. **Koblitz, N.** (1994). *A Course in Number Theory and Cryptography* (2nd ed.). Springer.

5. **Menezes, A. J., van Oorschot, P. C., & Vanstone, S. A.** (1996). *Handbook of Applied Cryptography*. CRC Press. Disponible gratis en: https://cacr.uwaterloo.ca/hac/

### Artículos originales

6. **Rivest, R. L., Shamir, A., & Adleman, L.** (1978). A method for obtaining digital signatures and public-key cryptosystems. *Communications of the ACM*, 21(2), 120–126.

7. **Rabin, M. O.** (1980). Probabilistic algorithm for testing primality. *Journal of Number Theory*, 12(1), 128–138.

8. **Miller, G. L.** (1976). Riemann's hypothesis and tests for primality. *Journal of Computer and System Sciences*, 13(3), 300–317.

### Recursos en línea

9. **Khan Academy** — Módulo aritmético: https://www.khanacademy.org/computing/computer-science/cryptography

10. **Cryptohack** — Plataforma de retos de criptografía (incluye aritmética modular): https://cryptohack.org

11. **NIST FIPS 186-5** (2023) — Estándar para generación de primos criptográficos: https://doi.org/10.6028/NIST.FIPS.186-5

12. **sympy.ntheory** — Módulo de teoría de números en Python: https://docs.sympy.org/latest/modules/ntheory.html

13. **The RSA Problem** — RSA Laboratories: https://www.rsa.com

### Videos recomendados

14. **3Blue1Brown** — "How RSA Cryptography Works" (YouTube)

15. **Computerphile** — Serie completa sobre criptografía (YouTube): búscar "Computerphile RSA"

---

> **Próxima clase:** Clase 3 — Criptografía de Clave Simétrica: AES, modos de operación, funciones hash y MAC.
